In [1]:
import os
import json
import pickle
import random
import numpy as np

# Path and file searching

In [2]:
# os.getcwd()

In [2]:
path_to_json_dataset_folder = "llm"

In [3]:
# find all related files first
raw_file_list = []

for root, dirs, files in os.walk(path_to_json_dataset_folder):
    for file in files:
        if file.lower().endswith('.json'):
            # only allowed formula types, no "negation" and "or"
            if  "type0000" in file.lower() or \
                "type0001" in file.lower() or \
                "type0002" in file.lower() or \
                "type0003" in file.lower() or \
                "type0004" in file.lower() or \
                "type0005" in file.lower() or \
                "type0006" in file.lower() or \
                "type0007" in file.lower() or \
                "type0008" in file.lower() or \
                "type0009" in file.lower() or \
                "type0010" in file.lower() or \
                "type0011" in file.lower() :
                
                # remove cache files from our list
                if ".ipynb_checkpoints" not in os.path.join(root, file):
#                     print(os.path.join(root, file))
                    raw_file_list.append(os.path.join(root, file))

In [4]:
raw_file_list

['llm/test_type0002_soft_efo1_qaa.json',
 'llm/test_type0006_soft_efo1_qaa.json',
 'llm/test_type0009_soft_efo1_qaa.json',
 'llm/test_type0001_soft_efo1_qaa.json',
 'llm/test_type0000_soft_efo1_qaa.json',
 'llm/test_type0010_soft_efo1_qaa.json',
 'llm/test_type0008_soft_efo1_qaa.json',
 'llm/test_type0005_soft_efo1_qaa.json',
 'llm/test_type0007_soft_efo1_qaa.json',
 'llm/test_type0011_soft_efo1_qaa.json',
 'llm/test_type0003_soft_efo1_qaa.json',
 'llm/test_type0004_soft_efo1_qaa.json']

In [5]:
# aggregation files into groups based on formula types
file_groups = {}
for t in ["type0000",
          "type0001",
          "type0002",
          "type0003",
          "type0004",
          "type0005",
          "type0006",
          "type0007",
          "type0008",
          "type0009",
          "type0010",
          "type0011"]:
    for file_path in raw_file_list:
        if t in file_path:
            if t not in file_groups:
                file_groups[t] = [file_path]
            else:
                file_groups[t].append(file_path)
        

In [6]:
file_groups

{'type0000': ['llm/test_type0000_soft_efo1_qaa.json'],
 'type0001': ['llm/test_type0001_soft_efo1_qaa.json'],
 'type0002': ['llm/test_type0002_soft_efo1_qaa.json'],
 'type0003': ['llm/test_type0003_soft_efo1_qaa.json'],
 'type0004': ['llm/test_type0004_soft_efo1_qaa.json'],
 'type0005': ['llm/test_type0005_soft_efo1_qaa.json'],
 'type0006': ['llm/test_type0006_soft_efo1_qaa.json'],
 'type0007': ['llm/test_type0007_soft_efo1_qaa.json'],
 'type0008': ['llm/test_type0008_soft_efo1_qaa.json'],
 'type0009': ['llm/test_type0009_soft_efo1_qaa.json'],
 'type0010': ['llm/test_type0010_soft_efo1_qaa.json'],
 'type0011': ['llm/test_type0011_soft_efo1_qaa.json']}

# Helper Functions

In [11]:
# find 4 unique f1 candidates, one with the max f1 value 
def should_include(f1_values, f1_answers):
    # return the first element for each unique value
    uniques, indices = np.unique(np.array(f1_values), return_index=True)
    
    # cannot distinguish max
    if len(uniques) < 2:
        return [], []
    elif len(indices) >= 4:
        non_answers = set(range(14999)).difference(set(f1_answers))
        left = 2
        selected_answers  = random.sample(non_answers, left)
        selected_values = [0.0 for _ in range(left)]
        part_answers = [f1_answers[indices[0]], f1_answers[indices[-1]]]
        part_values = [f1_values[indices[0]], f1_values[indices[-1]]]

        selected_answers.extend(part_answers)
        selected_values.extend(part_values)
        
        return selected_answers, selected_values
    else:
        non_answers = set(range(14999)).difference(set(f1_answers))
        left = 4 - len(uniques)
        selected_non_answers = random.sample(non_answers, left)
        part_answers = np.array(f1_answers)[indices].tolist()
        part_values = np.array(f1_values)[indices].tolist()
        selected_answers = selected_non_answers
        selected_values = [0.0 for _ in range(left)]
        selected_answers.extend(part_answers)
        selected_values.extend(part_values)
        
        # not sufficient
        
        return selected_answers, selected_values

# Load json

In [12]:
for t in ["type0000",
          "type0001",
          "type0002",
          "type0003",
          "type0004",
          "type0005",
          "type0006",
          "type0007",
          "type0008",
          "type0009",
          "type0010",
          "type0011"]:
    
    if t not in file_groups:
        continue
        
    curr_data_list = file_groups[t]

    idx = 0
    dataset = []

    for f in curr_data_list:
        qaa_data = json.load(open(f, "r"))
        lstr = list(qaa_data.keys())[0]
        raw_data = qaa_data[lstr]
        print(f)
        for query in raw_data:
            
            # one max f1_value + 3 lower values
            selected_f1_answes, selected_f1_values = should_include(query[2]["f1_values"], query[1]["f1_answers"])

            # no enough values for chatgpt  
            if len(selected_f1_answes) != 4:
                continue

            # format: (id, origin_string_query_type, dictionary , [4 candidate], [4 f1_values], correct_anwerer(max_candi))
            max_candidate = selected_f1_answes[np.argmax(selected_f1_values)]

            row = (idx, lstr, query[0], selected_f1_answes, selected_f1_values, max_candidate)
            dataset.append(row)
            idx = idx + 1
    pickle_file = open("llm/{}.pickle".format(t), "wb")
    pickle.dump(dataset, pickle_file)
    pickle_file.close()

llm/test_type0000_soft_efo1_qaa.json


/tmp/ipykernel_20267/1942191297.py:12: DeprecationWarning: Sampling from a set deprecated
since Python 3.9 and will be removed in a subsequent version.
  selected_answers  = random.sample(non_answers, left)
/tmp/ipykernel_20267/1942191297.py:24: DeprecationWarning: Sampling from a set deprecated
since Python 3.9 and will be removed in a subsequent version.
  selected_non_answers = random.sample(non_answers, left)
/tmp/ipykernel_20267/1942191297.py:24: DeprecationWarning: Sampling from a set deprecated
since Python 3.9 and will be removed in a subsequent version.
  selected_non_answers = random.sample(non_answers, left)
/tmp/ipykernel_20267/1942191297.py:12: DeprecationWarning: Sampling from a set deprecated
since Python 3.9 and will be removed in a subsequent version.
  selected_answers  = random.sample(non_answers, left)


llm/test_type0001_soft_efo1_qaa.json
llm/test_type0002_soft_efo1_qaa.json
llm/test_type0003_soft_efo1_qaa.json
llm/test_type0004_soft_efo1_qaa.json
llm/test_type0005_soft_efo1_qaa.json
llm/test_type0006_soft_efo1_qaa.json
llm/test_type0007_soft_efo1_qaa.json
llm/test_type0008_soft_efo1_qaa.json


/tmp/ipykernel_20267/1942191297.py:12: DeprecationWarning: Sampling from a set deprecated
since Python 3.9 and will be removed in a subsequent version.
  selected_answers  = random.sample(non_answers, left)
/tmp/ipykernel_20267/1942191297.py:24: DeprecationWarning: Sampling from a set deprecated
since Python 3.9 and will be removed in a subsequent version.
  selected_non_answers = random.sample(non_answers, left)


llm/test_type0009_soft_efo1_qaa.json
llm/test_type0010_soft_efo1_qaa.json
llm/test_type0011_soft_efo1_qaa.json


In [23]:
test = open("oct8/type0002.pickle","rb")
pickle.load(test)[:350]

[(0,
  '(r1(s1,f1,75%,0.4))&(r2(s2,f1,75%,0.8))',
  {'r1': 0, 'r2': 0, 's1': 1872, 's2': 5878},
  array([1952,   33, 2362, 8118]),
  array([0.8512, 0.9523, 0.8512, 0.8512]),
  33),
 (1,
  '(r1(s1,f1,25%,0.5))&(r2(s2,f1,25%,0.5))',
  {'r1': 8, 'r2': 8, 's1': 7699, 's2': 7699},
  array([ 497, 2351,  516, 1287]),
  array([0.7093, 1.    , 0.7093, 0.7093]),
  2351),
 (2,
  '(r1(s1,f1,25%,0.8))&(r2(s2,f1,75%,0.1))',
  {'r1': 0, 'r2': 0, 's1': 1583, 's2': 1583},
  array([ 730,  742, 1171, 1671]),
  array([0.6384, 0.8034, 0.6384, 0.6384]),
  742),
 (3,
  '(r1(s1,f1,25%,0.8))&(r2(s2,f1,25%,0.7))',
  {'r1': 3, 'r2': 3, 's1': 9075, 's2': 1290},
  array([1290, 4861, 3867, 6393]),
  array([1.064 , 1.1923, 1.064 , 1.064 ]),
  4861),
 (4,
  '(r1(s1,f1,75%,0.8))&(r2(s2,f1,25%,1.0))',
  {'r1': 0, 'r2': 0, 's1': 1258, 's2': 965},
  array([ 649, 4043, 1944, 2483]),
  array([1.1876, 1.2767, 1.3571, 1.3589]),
  2483),
 (5,
  '(r1(s1,f1,25%,0.4))&(r2(s2,f1,25%,0.4))',
  {'r1': 0, 'r2': 0, 's1': 5963, 's2': 